# Understand how Kalshi and Polymarket keep data

## Problem statement

Cross-platform arbitrage in prediction markets means finding the same real-world event listed on two different platforms when the prices do not agree. If Kalshi and Polymarket are describing the same outcome in different words, that price gap can be a signal.

The hard part is that there is no shared event ID, and the two platforms often phrase the same market very differently. This notebook is about turning that messy, mismatched text into comparable signals.

## 0: Import dependencies

In [28]:
import requests
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

## 1: Data sources

This notebook uses Kalshi and Polymarket market-discovery APIs directly. The important finding is that both platforms expose the market data we need publicly, without authentication, for discovery and analysis. Authentication only matters later if you want to place trades.

Docs used:
- Kalshi API docs: https://docs.kalshi.com/
- Polymarket API docs: https://docs.polymarket.com/

The next cells focus on politics markets so the matching problem stays concrete and easy to inspect.

### Category and tag filtering was not obvious

The first instinct was wrong in two places:
- Kalshi category filtering belongs on `/events`, not `/markets`.
- Polymarket politics filtering uses a numeric `tag_id`, not a text category name.

The working version below uses `category == "Politics"` on the Kalshi event feed and `tag_id=2` for Polymarket politics markets.

In [52]:
kalshi_market_raw = requests.get(
    "https://api.elections.kalshi.com/trade-api/v2/markets",
    params={"status": "open", "limit": 25},
).json()["markets"]

combo_markets = [m for m in kalshi_market_raw if m.get("mve_selected_legs")]
clean_markets = [m for m in kalshi_market_raw if not m.get("mve_selected_legs")]

print(f"Raw Kalshi markets: {len(kalshi_market_raw)}")
print(f"Clean Kalshi markets after filtering combos: {len(clean_markets)}")
print(f"Combo / multivariate markets filtered out: {len(combo_markets)}\n")

if combo_markets:
    example = combo_markets[0]
    print("Example combo market before filter:")
    print(f"- {example['title']}")
    print(f"- mve_selected_legs: {example['mve_selected_legs']}\n")
    print("After filtering, that market is removed from the matching pool.")
else:
    print("No combo markets showed up in this sample, but the filter is still applied.")

Raw Kalshi markets: 25
Clean Kalshi markets after filtering combos: 0
Combo / multivariate markets filtered out: 25

Example combo market before filter:
- yes Spain,yes Rory McIlroy,yes Tommy Fleetwood,yes France,yes Kylian Mbappe: 1+
- mve_selected_legs: [{'event_ticker': 'KXMENWORLDCUP-26', 'market_ticker': 'KXMENWORLDCUP-26-ES', 'side': 'yes'}, {'event_ticker': 'KXPGATOP20-THOC26', 'market_ticker': 'KXPGATOP20-THOC26-RMCI', 'side': 'yes'}, {'event_ticker': 'KXPGATOP20-THOC26', 'market_ticker': 'KXPGATOP20-THOC26-TFLE', 'side': 'yes'}, {'event_ticker': 'KXWCADVANCE-26JUL18FRAENG', 'market_ticker': 'KXWCADVANCE-26JUL18FRAENG-FRA', 'side': 'yes'}, {'event_ticker': 'KXWCGOAL-26JUL18FRAENG', 'market_ticker': 'KXWCGOAL-26JUL18FRAENG-FRAKMBAPP10-1', 'side': 'yes'}]

After filtering, that market is removed from the matching pool.


## 2: The messy data problem

The raw market feeds are not clean enough to use as-is. Kalshi has combo / multivariate structures that can look like ordinary markets at first glance, but they should be filtered out before semantic matching.

This section shows the raw data first, then the cleaned version so the difference is visible.

In [45]:
kalshi_events = requests.get(
    "https://api.elections.kalshi.com/trade-api/v2/events",
    params={"status": "open", "limit": 20, "with_nested_markets": "true"},
).json()["events"]

politics_events = [e for e in kalshi_events if e.get("category") == "Politics"]

kalshi_titles = []
for e in politics_events:
    for m in e.get("markets", []):
        kalshi_titles.append(m["title"])


In [46]:
poly_events = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"tag_id": 2, "active": "true", "closed": "false", "limit": 20},
).json()

poly_titles = []
for e in poly_events:
    for m in e.get("markets", []):
        poly_titles.append(m["question"])


In [47]:
print(f"Kalshi politics titles: {len(kalshi_titles)}")
for title in kalshi_titles[:5]:
    print(f"- {title}")


Kalshi politics titles: 41
- Will Ben Wikler be the next Chair of the Democratic National Committee?
- Will Jane Kleeb be the next Chair of the Democratic National Committee?
- Will Malcolm Kenyatta be the next Chair of the Democratic National Committee?
- Will Shasti Conrad be the next Chair of the Democratic National Committee?
- Will Reyna Walters-Morgan be the next Chair of the Democratic National Committee?


In [48]:
print(f"Polymarket politics titles: {len(poly_titles)}")
for title in poly_titles[:5]:
    print(f"- {title}")


Polymarket politics titles: 475
- Macron out in 2025?
- Macron out by July 31, 2026?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- China x India military clash by June 30?


## 4: Embed titles with a shared encoder

This is a shared-encoder plus ai pipeline: one sentence-transformer maps both Kalshi and Polymarket titles into the same vector space, then cosine similarity compares them. 

In [33]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5264.54it/s]


In [34]:
kalshi_embeddings = model.encode(kalshi_titles)
poly_embeddings = model.encode(poly_titles)

In [35]:
print(f"Shape of Kalshi embeddings: {kalshi_embeddings.shape}")
print(f"Shape of Polymarket embeddings: {poly_embeddings.shape}\n")

print("Example — first Kalshi title and its vector (first 10 dims):")
print(f"Title: {kalshi_titles[0]}")
print(f"Vector: {kalshi_embeddings[0][:10]} ... ({len(kalshi_embeddings[0])} dims total)")


Shape of Kalshi embeddings: (10, 384)
Shape of Polymarket embeddings: (10, 384)

Example — first Kalshi title and its vector (first 10 dims):
Title: Boston wins by over 21.5 points?
Vector: [ 0.00532584  0.02532064  0.02297051 -0.03077247 -0.00882336  0.01420813
  0.01732795  0.01935127  0.01510084  0.01370793] ... (384 dims total)


## 5: Compare cosine similarity

In [36]:
cosine_scores = util.cos_sim(kalshi_embeddings, poly_embeddings)

In [39]:
print(f"Similarity matrix shape: {cosine_scores.shape}")

Similarity matrix shape: torch.Size([10, 10])


In [40]:
THRESHOLD = 0.5

print("\n=== Best match per Kalshi market ===\n")
for i, k_title in enumerate(kalshi_titles):
    best_idx = cosine_scores[i].argmax().item()
    best_score = cosine_scores[i][best_idx].item()
    p_title = poly_titles[best_idx]

    flag = "✅ POSSIBLE MATCH" if best_score > THRESHOLD else "—"
    print(f"[{best_score:.3f}] {flag}")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}\n")


=== Best match per Kalshi market ===

[0.147] —
  Kalshi:     Boston wins by over 21.5 points?
  Polymarket: Will Harvey Weinstein be sentenced to between 10 and 20 years in prison?

[0.229] —
  Kalshi:     Milwaukee wins by over 24.5 points?
  Polymarket: Will Harvey Weinstein be sentenced to between 10 and 20 years in prison?

[0.090] —
  Kalshi:     Golden State wins by over 15.5 points
  Polymarket: Will China invades Taiwan before GTA VI?

[0.319] —
  Kalshi:     Will over 2.5 maps be played in the Zeu5 Esports vs. SDM Tigres League of Legends match?
  Polymarket: Will China invades Taiwan before GTA VI?

[0.251] —
  Kalshi:     Will over 2.5 maps be played in the Fuego vs. 3v Team League of Legends match?
  Polymarket: Will China invades Taiwan before GTA VI?

[0.267] —
  Kalshi:     Will over 2.5 maps be played in the Kits Esports vs. Polar Squad Esports League of Legends match?
  Polymarket: Will China invades Taiwan before GTA VI?

[0.060] —
  Kalshi:     Full Game: Over 204.

## 6: Trial: top cross-market matches

In [49]:
# Show the strongest matches across all Kalshi and Polymarket politics titles.
matches = []
for i in range(len(kalshi_titles)):
    for j in range(len(poly_titles)):
        matches.append((cosine_scores[i][j].item(), kalshi_titles[i], poly_titles[j]))

matches.sort(reverse=True, key=lambda x: x[0])

print("=== Top 10 matches across ALL pairs ===\n")
for score, k_title, p_title in matches[:10]:
    print(f"[{score:.3f}]")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}\n")


=== Top 10 matches across ALL pairs ===

[0.851]
  Kalshi:     Will Gretchen Whitmer be the next Chair of the Democratic National Committee?
  Polymarket: Will Gretchen Whitmer win the 2028 Democratic presidential nomination?

[0.849]
  Kalshi:     Will Rahm Emanuel be the next Chair of the Democratic National Committee?
  Polymarket: Will Rahm Emanuel win the 2028 Democratic presidential nomination?

[0.829]
  Kalshi:     Will Pete Buttigieg be the next Chair of the Democratic National Committee?
  Polymarket: Will Pete Buttigieg win the 2028 Democratic presidential nomination?

[0.823]
  Kalshi:     Will Tim Walz be the next Chair of the Democratic National Committee?
  Polymarket: Will Tim Walz win the 2028 Democratic presidential nomination?

[0.822]
  Kalshi:     Will Andy Beshear be the next Chair of the Democratic National Committee?
  Polymarket: Will Andy Beshear win the 2028 Democratic presidential nomination?

[0.817]
  Kalshi:     Will J.B. Pritzker be the next Chair of the

In [53]:
whitmer_example = next(
    (
        (score, k_title, p_title)
        for score, k_title, p_title in matches
        if "Whitmer" in k_title and "Whitmer" in p_title
    ),
    None,
)

if whitmer_example:
    score, k_title, p_title = whitmer_example
    print(f"[{score:.3f}] False positive example")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}")
else:
    print("Whitmer example not found in the current matches list.")

[0.851] False positive example
  Kalshi:     Will Gretchen Whitmer be the next Chair of the Democratic National Committee?
  Polymarket: Will Gretchen Whitmer win the 2028 Democratic presidential nomination?


## 7: Next step: add a verification layer

Cosine similarity is a good first pass, but the Whitmer example shows why it is not enough on its own. Similar names and related political topics can look like a match even when the actual outcomes are different.

The next step is a second-pass verifier, such as a Groq LLM check, that asks whether two questions are betting on the exact same real-world outcome. That extra pass should reduce false positives before anything is treated as a real arbitrage candidate.